#1. Importando Dataset

In [147]:
import pandas as pd

df = pd.read_json('TelecomX_Data.json')
for coluna in df.columns[2:]:
   df = df.drop(columns=[coluna]).join(pd.json_normalize(df[coluna]))

#2. Tratando as Inconsistências

In [148]:
df['Charges.Total'] = pd.to_numeric(df['Charges.Total'], errors='coerce')
df['SeniorCitizen'] = df['SeniorCitizen'].map({1: 'Yes', 0: 'No'})
df = df.dropna()

df = df[df['Churn'].str.strip() != '']
df['MultipleLines'] = df['MultipleLines'].str.replace(r"^No.*", "No", regex=True)
for coluna in df.columns[10:16]:
  df[coluna] = df[coluna].str.replace(r"^No.*", "No", regex=True)

#3. Criando Novas Tabelas

In [150]:
df['Contas_Diarias'] = df['Charges.Monthly'] / 30

#4. Análise Descritiva

In [162]:
analise_geral = df.groupby("Churn")[["tenure", "Charges.Monthly", "Charges.Total", "Contas_Diarias"]].agg(["mean", "median", "std"])

Churn,No,Yes
OnlineSecurity,85.359801,14.640199
OnlineBackup,78.432990,21.567010
DeviceProtection,77.460711,22.539289
TechSupport,84.803922,15.196078
StreamingTV,69.885313,30.114687
StreamingMovies,70.047602,29.952398


#5. Distribuição da Evasão

##a. Cálculos

In [ ]:
percentual_cancelamento = df["Churn"].value_counts(normalize=True) * 100
cancelamentos_contrato = pd.crosstab(df["Contract"], df["Churn"], normalize="index") * 100

resultados = []
for servico in df.columns[10:16]:
    tabela = pd.crosstab(df[servico], df["Churn"], normalize="index") * 100

    if "Yes" in tabela.index:
        linha = tabela.loc["Yes"]
        linha.name = servico
        resultados.append(linha)

analise_servicos = pd.DataFrame(resultados)

##b. Gráficos